In [1]:
import torch
import torch.nn as nn

In [2]:
from slowfast.models.stem_helper import ResNetBasicStem
from slowfast.models.video_model_builder import FuseFastToSlow

test ResNetBasicStem
## slow path

In [10]:
# 위에서 정의한 ResNetBasicStem 클래스 사용
stem = ResNetBasicStem(
    dim_in=3,          # 입력 채널: RGB 영상
    dim_out=64,        # 출력 채널: 보통 ResNet stem은 64
    kernel=[1, 7, 7],  # 3D conv kernel: temporal=3, height=7, width=7
    stride=[1, 2, 2],  # 시간축 stride=1, 공간축 stride=2
    padding=[0, 3, 3], # 시간축 padding=1, 공간축 padding=3
)

In [11]:
# 가짜 입력 데이터 생성
# 크기: (Batch, Channel, Time, Height, Width)
x_s = torch.randn(2, 3, 4, 224, 224)  
print("입력 크기:", x_s.shape)

입력 크기: torch.Size([2, 3, 4, 224, 224])


In [12]:
# forward
out = stem.conv(x_s)
print("출력 크기:", out.shape)

출력 크기: torch.Size([2, 64, 4, 112, 112])


## fast path

In [18]:
stem = ResNetBasicStem(
    dim_in=3,          # 입력 채널: RGB 영상
    dim_out=8,        # 출력 채널: 보통 ResNet stem은 64
    kernel=[5, 7, 7],  # 3D conv kernel: temporal=3, height=7, width=7
    stride=[1, 2, 2],  # 시간축 stride=1, 공간축 stride=2
    padding=[2, 3, 3], # 시간축 padding=1, 공간축 padding=3
)

x_f = torch.randn(2, 3, 32, 224, 224)  
print("입력 크기:", x_f.shape)

out = stem.conv(x_f)
print("conv1_출력 크기:", out.shape)

out_1 = stem.pool_layer(out)
print("pool1_출력 크기:", out_1.shape)

입력 크기: torch.Size([2, 3, 32, 224, 224])
conv1_출력 크기: torch.Size([2, 8, 32, 112, 112])
pool1_출력 크기: torch.Size([2, 8, 32, 56, 56])


In [14]:
(112+2*1-3)/2

55.5

In [ ]:
112+2*3-7-1

In [ ]:
57/2

In [19]:
N=2            # 배치 크기
C_slow=64      # Slow 경로 채널
C_fast=8       # Fast 경로 채널 (FuseFastToSlow의 dim_in)
T_slow=8       # Slow 시간 길이
H=56
W=56     # 공간 크기
alpha=8        # Fast/Slow 프레임 비율
fusion_ratio=2 # fusion_conv_channel_ratio
fusion_kernel=5

In [20]:
device = "cpu"  # 필요하면 "cuda"로 변경
torch.manual_seed(7)

# 더미 입력 생성
# Fast는 시간축이 alpha배 길어야 함
T_fast = alpha * T_slow
x_s = torch.randn(N, C_slow, T_slow, H, W, requires_grad=True, device=device)
x_f = torch.randn(N, C_fast, T_fast, H, W, requires_grad=True, device=device)

In [24]:
print(f"x_s.shape:{x_s.shape}, x_f.shape:{x_f.shape}")

x_s.shape:torch.Size([2, 64, 8, 56, 56]), x_f.shape:torch.Size([2, 8, 64, 56, 56])


In [21]:
# 모듈 생성
fuse_block = FuseFastToSlow(
    dim_in=C_fast,
    fusion_conv_channel_ratio=fusion_ratio,
    fusion_kernel=fusion_kernel,
    alpha=alpha,
).to(device)

In [26]:
x_f_out = fuse_block.conv_f2s(x_f)

In [28]:
(64+4-5)/8

7.875

In [27]:
x_f_out.shape

torch.Size([2, 16, 8, 56, 56])

In [25]:
# 순전파
x_s_fuse, x_f_out = fuse_block([x_s, x_f])


# 모양 확인
print("Input Slow :", tuple(x_s.shape))
print("Input Fast :", tuple(x_f.shape))
print("Output Slow(fused):", tuple(x_s_fuse.shape))
print("Output Fast (pass):", tuple(x_f_out.shape))

Input Slow : (2, 64, 8, 56, 56)
Input Fast : (2, 8, 64, 56, 56)
Output Slow(fused): (2, 80, 8, 56, 56)
Output Fast (pass): (2, 8, 64, 56, 56)


In [ ]:
x_f.shape

In [ ]:
x_f_out = fuse_block.conv_f2s(x_f)

In [ ]:
x_f_out.shape

In [ ]:
x_f_out1 = fuse_block.bn(x_f_out)

In [ ]:
x_f_out1.shape

In [ ]:
x_f_out1 = fuse_block.relu(x_f_out1)

In [ ]:
x_f_out1.shape 

In [ ]:
x_s.shape

In [ ]:
 x_s_fuse = torch.cat([x_s, x_f_out1], 1)

In [ ]:
 x_s_fuse.shape

In [ ]:
(64+2*2-4-1)/4


In [ ]:
5//2